# 6 &middot; Why Line Search Matters

A Newton *direction* \(-H^{-1}\nabla E\) points the right way, but the *step
length* still matters. Too large a step overshoots, inverts elements, and the
energy **explodes** instead of decreasing.

**Backtracking line search** starts from a full step and shrinks it until the
energy actually decreases (the Armijo condition). To make the failure easy to
watch, we move the handle **slowly right and back** and take a few Newton
iterations per frame &mdash; once with a fixed, too-large step, once with line
search.

In [1]:
# --- make the *local* simkit (this repo) importable, ahead of any installed copy ---
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
%matplotlib inline
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import simkit
import simkit.energies as energies
from simkit.solvers.NewtonSolver import NewtonSolver, NewtonSolverParams
import utils
os.makedirs("media", exist_ok=True)

## A thick, stiff, nearly-incompressible beam

Same objective as tutorial 5 but **thicker** (so bending stores more energy) and
**nearly incompressible** (\(\nu = 0.49\) &mdash; resisting volume change makes
the energy much stiffer), so an over-eager step bites fast and the value of line
search is obvious. The handle (the full right edge) follows a there-and-back
ramp.

In [2]:
X, T = utils.triangulated_grid(nx=16, ny=7, width=2.0, height=0.6)   # thick beam
n, dim = X.shape
J, vol = simkit.deformation_jacobian(X, T), simkit.volume(X, T)
mu, lam = simkit.ympr_to_lame(50.0, 0.49)                            # high Poisson ratio
mu  = np.full((len(T), 1), mu)
lam = np.full((len(T), 1), lam)

pin_idx   = np.where(X[:, 0] <= X[:, 0].min() + 1e-6)[0]
right_idx = np.where(X[:, 0] >= X[:, 0].max() - 1e-6)[0]
Q_pin, b_pin = simkit.dirichlet_penalty(pin_idx, X[pin_idx], n, 1e6)

def quad(Q, b, xc):  return 0.5 * (xc.T @ (Q @ xc))[0, 0] + (b.T @ xc)[0, 0]

def make_objective(Q_h, b_h):
    def energy(x):
        xn, xc = x.reshape(-1, dim), x.reshape(-1, 1)
        E_elastic = energies.macklin_mueller_neo_hookean_energy_x(xn, J, mu, lam, vol)
        E_pin     = quad(Q_pin, b_pin, xc)
        E_handle  = quad(Q_h, b_h, xc)
        E_total   = E_elastic + E_pin + E_handle
        return E_total
    def gradient(x):
        xn, xc = x.reshape(-1, dim), x.reshape(-1, 1)
        g_elastic = energies.macklin_mueller_neo_hookean_gradient_x(xn, J, mu, lam, vol)
        g_pin     = Q_pin @ xc + b_pin
        g_handle  = Q_h @ xc + b_h
        g_total   = g_elastic + g_pin + g_handle
        return g_total
    def hessian(x):
        xn = x.reshape(-1, dim)
        H_elastic = energies.macklin_mueller_neo_hookean_hessian_x(xn, J, mu, lam, vol, psd=True)
        H_pin     = Q_pin
        H_handle  = Q_h
        H_total   = H_elastic + H_pin + H_handle
        return H_total
    return energy, gradient, hessian

ramp = np.concatenate([np.linspace(0, 1, 22), np.linspace(1, 0, 22)])    # right then back
offsets = [np.array([1.0 * r, 0.0]) for r in ramp]

## The quasi-static driver

At each handle position we take a few Newton iterations. The only difference
between the two runs is the step: a fixed `STEP` (deliberately too big) versus
the backtracking step that guarantees the energy decreases.

In [3]:
DIVERGE = 1e3

def drive(use_line_search, STEP=2.0, iters_per_frame=2):
    x = X.flatten().reshape(-1, 1).copy()
    states, blew_at = [], None
    for k, off in enumerate(offsets):
        Q_h, b_h = simkit.dirichlet_penalty(right_idx, X[right_idx] + off, n, 1e6)
        energy, gradient, hessian = make_objective(Q_h, b_h)
        for _ in range(iters_per_frame):
            g, H = gradient(x), hessian(x)
            dx = sp.sparse.linalg.spsolve(H.tocsc(), -g).reshape(-1, 1)
            alpha = simkit.backtracking_line_search(energy, x, g, dx)[0] if use_line_search else STEP
            x = x + alpha * dx
        states.append(x.reshape(n, dim).copy())
        if not np.isfinite(np.abs(x).max()) or np.abs(x).max() > DIVERGE:
            blew_at = k; break
    return states, blew_at

fixed_states, blew = drive(use_line_search=False, STEP=2.0)
ls_states,    _    = drive(use_line_search=True)
print("fixed step (no line search): blew up at frame", blew, "of", len(offsets))
print("line search: completed all", len(ls_states), "frames")

fixed step (no line search): blew up at frame 9 of 44
line search: completed all 44 frames


## Without line search: the solver explodes (fixed step = 2x the Newton step)

In [4]:
lims = ((-1.3, 2.2), (-1.6, 1.6))
fig, anim = utils.animate_mesh(fixed_states, T, lims=lims,
    title="Fixed step, NO line search  ->  explodes", pin_pts=X[pin_idx],
    handle_traj=[s[right_idx] for s in fixed_states], fps=12)
utils.save_anim(anim, "media/06_no_line_search.mp4", fps=12)
plt.close(fig)
utils.show_anim(anim)

## With line search: smooth and stable

In [5]:
fig, anim = utils.animate_mesh(ls_states, T, lims=lims,
    title="Backtracking line search  ->  stable", pin_pts=X[pin_idx],
    handle_traj=[s[right_idx] for s in ls_states], fps=20)
utils.save_anim(anim, "media/06_line_search.mp4", fps=20)
plt.close(fig)
utils.show_anim(anim)

### Takeaways
* A good Newton *direction* isn't enough &mdash; an unchecked **step length** can
  diverge.
* **Backtracking line search** shrinks the step until the energy decreases.
* Cheap insurance: a few extra energy evaluations buy a solver that won't blow up.